In [ ]:
# CELL 1 — INSTALL  (run once, restart kernel after)
!pip install kymatio PyWavelets --quiet
print("✅ kymatio + PyWavelets installed")

In [ ]:
# CELL 2 — IMPORTS
import os, pickle, warnings, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pywt                         # PyWavelets — CWT / DWT / SWT
from kymatio.torch import Scattering1D   # Scattering wavelet transform (GPU)

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white',
                     'figure.dpi':110,'savefig.dpi':150})
print(f"Device: {DEVICE}")
print("✅ Imports ready")

In [ ]:
# CELL 3 — PATHS & CONFIG  (Colab version)
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/his/HIS Project'  # ← adjust to your actual folder name

SAVE_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
CKPT_DIR = os.path.join(PROJECT_ROOT, 'reports', 'checkpoints')
FIG_DIR  = os.path.join(PROJECT_ROOT, 'reports', 'figures', 'wavelet')
FEAT_DIR = SAVE_DIR
os.makedirs(FIG_DIR, exist_ok=True)

with open(os.path.join(SAVE_DIR, 'config.pkl'), 'rb') as f:
    cfg = pickle.load(f)

FS        = cfg['sampling_rate']
INPUT_LEN = cfg['input_len']
HORIZON   = cfg['horizon']
N_LEADS   = cfg['n_leads']
LEAD_NAMES= cfg['lead_names']

print(f"Input   : {INPUT_LEN} samples @ {FS} Hz")
print(f"Horizon : {HORIZON} samples")
print(f"Leads   : {LEAD_NAMES}")
print("✅ Config loaded")

In [ ]:
# CELL 4 — LOAD DATA
X_train = np.load(os.path.join(SAVE_DIR, 'X_train.npy'))   # (N, T, 12)
y_train = np.load(os.path.join(SAVE_DIR, 'y_train.npy'))
X_val   = np.load(os.path.join(SAVE_DIR, 'X_val.npy'))
y_val   = np.load(os.path.join(SAVE_DIR, 'y_val.npy'))
X_test  = np.load(os.path.join(SAVE_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(SAVE_DIR, 'y_test.npy'))

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print("✅ Data loaded")

## Section 1: Frequency Transformation Theory
---
See markdown below.

In [ ]:
"""
# 1.1  Short-Time Fourier Transform (STFT)
-----------------------------------------
The STFT computes the Fourier spectrum inside a sliding window:

    STFT{x}(t, f) = ∫ x(τ) · g*(τ−t) · e^{−j2πfτ} dτ

where g is a window function (Hann, Hamming, …).
Limitation: fixed time-frequency resolution — wide window → fine frequency, poor time;
narrow window → fine time, poor frequency (Heisenberg uncertainty).

# 1.2  Continuous Wavelet Transform (CWT)
-----------------------------------------
The CWT replaces the Fourier sinusoid with a *wavelet* ψ that is
jointly localised in time AND scale (frequency):

    CWT{x}(a, b) = (1/√a) · ∫ x(t) · ψ*((t−b)/a) dt

- a = scale (large a → low frequency, stretched wavelet)
- b = translation (time shift)

# 1.3  Discrete Wavelet Transform (DWT)
----------------------------------------
Discrete version using dyadic scales a = 2^j, computed via filterbanks.

# 1.4  Stationary (Undecimated) Wavelet Transform (SWT)
---------------------------------------------------------
Like DWT but without downsampling — all coefficient arrays have the same length as input.

# 1.5  Common Wavelet Families
-------------------------------
| Family     | Orthogonal | Support | Regularity | ECG Use                      |
|------------|------------|---------|------------|------------------------------|
| Haar       | Yes        | 2       | Low        | Edge detection               |
| Daubechies | Yes        | 2N−1    | Medium     | QRS detection (db4/db6)      |
| Symlets    | Yes        | 2N−1    | High       | Smooth ECG feature extraction|
| Coiflets   | Yes        | 6N−1    | High       | Baseline wander removal      |
| Morlet     | No (CWT)   | ∞       | High       | Time-frequency representation|

# 1.6  Scattering Wavelet Transform (the one we USE)
------------------------------------------------------
Proposed by Mallat (2012).  Produces *stable, invariant* representations
by alternating wavelet convolutions with non-linear modulus + averaging.

Properties critical for ECG:
  • Translation invariance
  • Deformation stability
  • Energy preservation
  • Deterministic — no training required for the wavelet stage
"""

print("Section 1 theory displayed above — read the docstring for mathematical detail.")

## Section 2: Wavelet Family Visualisation on ECG Signal

In [ ]:
# CELL 5 — CWT SCALOGRAM  (Morlet)
sample_idx = 0
lead_idx   = 1   # Lead II — clearest P-QRS-T

sig = X_train[sample_idx, :, lead_idx]   # (INPUT_LEN,)
t   = np.arange(len(sig)) / FS

# CWT via PyWavelets (morlet)
scales   = np.geomspace(1, 64, num=64)
coeffs, freqs = pywt.cwt(sig, scales, 'morl', sampling_period=1/FS)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios':[1,2]})
axes[0].plot(t, sig, color='#2c3e50', lw=1.5)
axes[0].set_ylabel('mV'); axes[0].set_title(f'Lead {LEAD_NAMES[lead_idx]} — Input Signal', fontweight='bold')
axes[0].grid(alpha=0.3)

im = axes[1].pcolormesh(t, freqs, np.abs(coeffs), cmap='hot', shading='gouraud')
axes[1].set_ylabel('Frequency (Hz)', fontsize=11)
axes[1].set_xlabel('Time (s)', fontsize=11)
axes[1].set_yscale('log')
axes[1].set_ylim(freqs[-1], freqs[0])
axes[1].set_title('CWT Scalogram (Morlet) — Time-Frequency Power', fontweight='bold')
plt.colorbar(im, ax=axes[1], label='|CWT| magnitude')

plt.suptitle(f'Continuous Wavelet Transform — PTB-XL ECG  (Sample {sample_idx}, Lead {LEAD_NAMES[lead_idx]})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_cwt_scalogram.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ 01_cwt_scalogram.png saved")

In [ ]:
# CELL 6 — DWT DECOMPOSITION  (db4, 5 levels)
wavelet = 'db4'
max_level = 5
sig = X_train[sample_idx, :, lead_idx]

coeffs = pywt.wavedec(sig, wavelet, level=max_level)
# coeffs = [cA5, cD5, cD4, cD3, cD2, cD1]

labels = [f'cA{max_level}'] + [f'cD{max_level-i}' for i in range(max_level)]
colors = ['#2980b9'] + ['#e74c3c','#e67e22','#27ae60','#9b59b6','#1abc9c']

fig, axes = plt.subplots(len(coeffs)+1, 1, figsize=(14, 2.5*(len(coeffs)+1)))
axes[0].plot(t, sig, color='#2c3e50', lw=1.5)
axes[0].set_title(f'Original — Lead {LEAD_NAMES[lead_idx]}', fontweight='bold')
axes[0].grid(alpha=0.3)

for i, (c, lbl, col) in enumerate(zip(coeffs, labels, colors)):
    t_c = np.linspace(0, INPUT_LEN/FS, len(c))
    axes[i+1].plot(t_c, c, color=col, lw=1.5)
    bandwidth = f'{FS/2**(i+1):.1f}–{FS/2**i:.1f} Hz' if i>0 else f'<{FS/2**max_level:.1f} Hz'
    axes[i+1].set_title(f'{lbl}  ({bandwidth})  len={len(c)}', fontweight='bold')
    axes[i+1].grid(alpha=0.3)
    for ax in axes: ax.set_ylabel('mV')

axes[-1].set_xlabel('Time (s)')
plt.suptitle(f'DWT Decomposition ({wavelet}, {max_level} levels) — Lead {LEAD_NAMES[lead_idx]}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_dwt_decomposition.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ 02_dwt_decomposition.png saved")

In [ ]:
# CELL 7 — WAVELET FAMILY COMPARISON (energy in detail bands)
families = ['db4', 'sym5', 'coif3', 'haar']
colors   = ['#e74c3c', '#2ecc71', '#3498db', '#e67e22']
level    = 5

fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(level+1)

for fam, col in zip(families, colors):
    c = pywt.wavedec(sig, fam, level=level)
    energy = [np.sum(arr**2) for arr in c]
    total  = sum(energy) + 1e-12
    ax.plot(x_pos, [e/total*100 for e in energy], 'o-', lw=2, color=col,
            label=fam, markersize=8)

ax.set_xticks(x_pos)
ax.set_xticklabels([f'cA{level}']+[f'cD{level-i}' for i in range(level)], fontsize=10)
ax.set_xlabel('Decomposition Level', fontsize=11)
ax.set_ylabel('Energy (% of total)', fontsize=11)
ax.set_title('Wavelet Family Comparison — Energy Distribution Across DWT Bands',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_wavelet_family_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ 03_wavelet_family_comparison.png saved")

## Section 3: Scattering Wavelet Transform on PTB-XL Dataset

In [ ]:
# CELL 8 — SCATTERING TRANSFORM SETUP  (Colab GPU version)
from kymatio.torch import Scattering1D   # ← torch backend, NOT kymatio.numpy

J = 6
Q = 8
scat = Scattering1D(J=J, shape=INPUT_LEN, Q=Q).to(DEVICE)   # ← .to(DEVICE)

# Dry run to get output shape
with torch.no_grad():
    x_dummy = torch.zeros(1, INPUT_LEN, device=DEVICE)
    Sx_dummy = scat(x_dummy)

N_COEF = Sx_dummy.shape[-2]
N_SCAT = Sx_dummy.shape[-1]
SCAT_DIM = N_COEF * N_LEADS

print(f"Scattering1D  J={J}  Q={Q}  input_len={INPUT_LEN}")
print(f"Output per sample per lead: ({N_COEF}, {N_SCAT})")
print(f"Static feature dim: {SCAT_DIM}")
print("✅ Scattering transform ready (GPU)")

In [ ]:
# CELL 9 — SCATTERING FEATURE EXTRACTION  (Colab GPU version)
def extract_scattering_static(X_np, scat_op, batch=64, desc=''):
    N = len(X_np)
    all_feats = []
    for start in range(0, N, batch):
        X_b = X_np[start:start+batch]         # (B, T, 12)
        B   = len(X_b)
        lead_feats = []
        for li in range(X_b.shape[2]):
            x_lead = torch.tensor(X_b[:, :, li], dtype=torch.float32, device=DEVICE)
            with torch.no_grad():
                Sx = scat_op(x_lead)           # (B, C, T_out)  — GPU tensor
            lead_feats.append(Sx.mean(dim=-1).cpu().numpy())   # (B, C)
        all_feats.append(np.concatenate(lead_feats, axis=1))   # (B, C*12)
        if (start // batch) % 10 == 0:
            print(f"  {desc}  {start}/{N}", end='\r')
    print()
    return np.concatenate(all_feats, axis=0)

print("Extracting scattering features — train …")
t0 = time.time()
scat_train = extract_scattering_static(X_train, scat, desc='train')
print(f"  Done in {time.time()-t0:.1f}s  shape: {scat_train.shape}")

print("Extracting scattering features — val …")
scat_val   = extract_scattering_static(X_val,   scat, desc='val')
print(f"  shape: {scat_val.shape}")

print("Extracting scattering features — test …")
scat_test  = extract_scattering_static(X_test,  scat, desc='test')
print(f"  shape: {scat_test.shape}")

SCAT_DIM = scat_train.shape[1]
print(f"\nStatic scattering feature dim per sample: {SCAT_DIM}")
print("✅ Scattering features extracted")

In [ ]:
# CELL 10 — SAVE SCATTERING FEATURES
np.save(os.path.join(FEAT_DIR, 'scat_train.npy'), scat_train.astype(np.float32))
np.save(os.path.join(FEAT_DIR, 'scat_val.npy'),   scat_val.astype(np.float32))
np.save(os.path.join(FEAT_DIR, 'scat_test.npy'),  scat_test.astype(np.float32))

# Normalise (fit on train only)
scat_mu  = scat_train.mean(0, keepdims=True)
scat_sig = scat_train.std(0, keepdims=True) + 1e-8

scat_train_n = (scat_train - scat_mu) / scat_sig
scat_val_n   = (scat_val   - scat_mu) / scat_sig
scat_test_n  = (scat_test  - scat_mu) / scat_sig

np.save(os.path.join(FEAT_DIR, 'scat_train_norm.npy'), scat_train_n.astype(np.float32))
np.save(os.path.join(FEAT_DIR, 'scat_val_norm.npy'),   scat_val_n.astype(np.float32))
np.save(os.path.join(FEAT_DIR, 'scat_test_norm.npy'),  scat_test_n.astype(np.float32))

scat_meta = {'scat_dim': SCAT_DIM, 'J': J, 'Q': Q, 'INPUT_LEN': INPUT_LEN,
             'mu': scat_mu, 'sigma': scat_sig}
with open(os.path.join(FEAT_DIR, 'scat_meta.pkl'), 'wb') as f:
    pickle.dump(scat_meta, f)

print(f"Saved normalised scattering features → {FEAT_DIR}")
print(f"scat_train_norm : {scat_train_n.shape}")
print("✅ Features saved")

In [ ]:
# CELL 11 — SCATTERING COEFFICIENT VISUALISATION  (Colab GPU version)
sig_lead = torch.tensor(X_train[sample_idx, :, lead_idx][np.newaxis, :],
                        dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    Sx = scat(sig_lead).cpu().numpy()[0]   # (C, T_out)

t_sc = np.linspace(0, INPUT_LEN/FS, Sx.shape[1])

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(t, X_train[sample_idx, :, lead_idx], color='#2c3e50', lw=1.5)
axes[0].set_title(f'Lead {LEAD_NAMES[lead_idx]} — Original Signal', fontweight='bold')
axes[0].set_ylabel('mV'); axes[0].grid(alpha=0.3)

im = axes[1].pcolormesh(t_sc, np.arange(Sx.shape[0]), Sx,
                        cmap='viridis', shading='gouraud')
axes[1].set_xlabel('Time (s)', fontsize=11)
axes[1].set_ylabel('Scattering coefficient index', fontsize=11)
axes[1].set_title(f'Scattering Coefficients (J={J}, Q={Q}) — {Sx.shape[0]} coefficients',
                  fontweight='bold')
plt.colorbar(im, ax=axes[1], label='Coefficient value')

plt.suptitle('Scattering Wavelet Transform — PTB-XL ECG', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04_scattering_coefficients.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ 04_scattering_coefficients.png saved")

## Section 4: Injecting Scattering Features into the Model Pipeline

In [ ]:
# CELL 12 — ECGUNet + Scattering Injection (FiLM conditioning)
class ConvBNGELU(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, d=1):
        super().__init__()
        pad = ((k-1)*d)//2
        self.net = nn.Sequential(
            nn.Conv1d(in_c, out_c, k, stride=s, dilation=d, padding=pad, bias=False),
            nn.GroupNorm(min(8,out_c), out_c), nn.GELU())
    def forward(self, x): return self.net(x)

class EncoderBlock(nn.Module):
    def __init__(self, in_c, out_c, dilation=1):
        super().__init__()
        self.conv1=ConvBNGELU(in_c,out_c,k=5,d=dilation)
        self.conv2=ConvBNGELU(out_c,out_c,k=5,d=dilation)
        self.down=nn.Conv1d(out_c,out_c,2,stride=2,bias=False)
        self.drop=nn.Dropout(0.10)
    def forward(self,x):
        x=self.conv2(self.conv1(x)); skip=x
        return self.drop(self.down(x)), skip

class DecoderBlock(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up=nn.ConvTranspose1d(in_c,in_c,2,stride=2)
        self.conv1=ConvBNGELU(in_c+skip_c,out_c,k=5)
        self.conv2=ConvBNGELU(out_c,out_c,k=5)
        self.drop=nn.Dropout(0.10)
    def forward(self,x,skip):
        x=self.up(x)
        diff=skip.shape[-1]-x.shape[-1]
        if diff>0: x=F.pad(x,(0,diff))
        elif diff<0: x=x[...,:skip.shape[-1]]
        return self.drop(self.conv2(self.conv1(torch.cat([x,skip],dim=1))))

class ECGUNetScat(nn.Module):
    def __init__(self, n_leads=12, horizon=490, scat_dim=None, dropout=0.10):
        super().__init__()
        self.horizon = horizon
        self.enc1=EncoderBlock(n_leads,  64, dilation=1)
        self.enc2=EncoderBlock(64,      128, dilation=2)
        self.enc3=EncoderBlock(128,     256, dilation=4)
        self.enc4=EncoderBlock(256,     256, dilation=8)
        self.bottleneck_lstm=nn.LSTM(256,128,2,batch_first=True,bidirectional=True,dropout=dropout)
        self.bottleneck_proj=ConvBNGELU(256,256,k=1)
        self.film = None
        if scat_dim is not None:
            self.film = nn.Sequential(
                nn.Linear(scat_dim, 256), nn.GELU(),
                nn.Linear(256, 512))
        self.dec4=DecoderBlock(256,256,256); self.dec3=DecoderBlock(256,256,128)
        self.dec2=DecoderBlock(128,128, 64); self.dec1=DecoderBlock( 64, 64, 32)
        self.out_conv=nn.Conv1d(32,n_leads,1)

    def forward(self, x, scat_feat=None):
        x,s1=self.enc1(x); x,s2=self.enc2(x); x,s3=self.enc3(x); x,s4=self.enc4(x)
        lo,_=self.bottleneck_lstm(x.permute(0,2,1))
        x=self.bottleneck_proj(lo.permute(0,2,1))
        if self.film is not None and scat_feat is not None:
            film_out = self.film(scat_feat)
            gamma = film_out[:, :256].unsqueeze(-1) + 1
            beta  = film_out[:, 256:].unsqueeze(-1)
            x = x * gamma + beta
        x=self.dec4(x,s4); x=self.dec3(x,s3); x=self.dec2(x,s2); x=self.dec1(x,s1)
        x=x[...,:self.horizon]
        return self.out_conv(x).permute(0,2,1)

_m = ECGUNetScat(N_LEADS, HORIZON, scat_dim=SCAT_DIM).to(DEVICE)
with torch.no_grad():
    _x  = torch.randn(4, N_LEADS, INPUT_LEN).to(DEVICE)
    _sf = torch.randn(4, SCAT_DIM).to(DEVICE)
    _o  = _m(_x, _sf)
    assert _o.shape == (4, HORIZON, N_LEADS), f"Wrong output shape: {_o.shape}"
    print(f"Forward test: {tuple(_x.shape)} + scat({SCAT_DIM}) → {tuple(_o.shape)}")
n_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Parameters: {n_p:,}")
del _m
print("✅ ECGUNetScat defined and tested")

In [ ]:
# CELL 13 — DATASET & TRAINING LOOP (with scattering features)
def fit_norm(X):
    vals = X.ravel(); mu,si = vals.mean(), vals.std()
    return mu, si, (X-mu)/(si+1e-8)

mu_x, si_x, X_tr_n = fit_norm(X_train)
_, _, X_vl_n = (lambda m,s,x: (m,s,(x-m)/(s+1e-8)))(mu_x, si_x, X_val)
mu_y, si_y, y_tr_n = fit_norm(y_train)
_, _, y_vl_n = (lambda m,s,x: (m,s,(x-m)/(s+1e-8)))(mu_y, si_y, y_val)

X_tr_t  = torch.tensor(X_tr_n.transpose(0,2,1), dtype=torch.float32)
y_tr_t  = torch.tensor(y_tr_n,                   dtype=torch.float32)
sf_tr_t = torch.tensor(scat_train_n,             dtype=torch.float32)
X_vl_t  = torch.tensor(X_vl_n.transpose(0,2,1), dtype=torch.float32)
y_vl_t  = torch.tensor(y_vl_n,                   dtype=torch.float32)
sf_vl_t = torch.tensor(scat_val_n,               dtype=torch.float32)

BS = 64
tr_ds = TensorDataset(X_tr_t, y_tr_t, sf_tr_t)
vl_ds = TensorDataset(X_vl_t, y_vl_t, sf_vl_t)
tr_dl = DataLoader(tr_ds, batch_size=BS, shuffle=True,  drop_last=True, num_workers=0)
vl_dl = DataLoader(vl_ds, batch_size=BS, shuffle=False, drop_last=False, num_workers=0)
print(f"Train batches: {len(tr_dl)}  Val batches: {len(vl_dl)}")

def composite_loss(pred, target):
    huber = F.huber_loss(pred, target, delta=0.5)
    grad_p = pred[:, 1:] - pred[:, :-1]
    grad_t = target[:, 1:] - target[:, :-1]
    grad_l = (grad_p - grad_t).abs().mean()
    return huber + 3.0 * grad_l

print("✅ DataLoaders and loss ready")

In [ ]:
# CELL 14 — TRAIN ECGUNetScat
from tqdm.notebook import tqdm   # nicer progress bars in Colab

model_scat = ECGUNetScat(N_LEADS, HORIZON, scat_dim=SCAT_DIM, dropout=0.10).to(DEVICE)
n_p = sum(p.numel() for p in model_scat.parameters() if p.requires_grad)
print(f"ECGUNetScat  params: {n_p:,}")

optimizer = torch.optim.AdamW(model_scat.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20)

CKPT_PATH = os.path.join(CKPT_DIR, 'ECGUNetScat_best.pt')
best_val   = float('inf')
patience   = 20
no_improve = 0
history    = {'train':[], 'val':[], 'lr':[]}
N_EPOCHS   = 80

pbar = tqdm(range(1, N_EPOCHS+1), desc='Epochs', ncols=90)
for ep in pbar:
    model_scat.train(); tr_losses = []
    for xb, yb, sb in tr_dl:
        xb=xb.to(DEVICE); yb=yb.to(DEVICE); sb=sb.to(DEVICE)
        optimizer.zero_grad()
        pred = model_scat(xb, sb)
        loss = composite_loss(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model_scat.parameters(), 0.5)
        optimizer.step()
        tr_losses.append(loss.item())
    scheduler.step()

    model_scat.eval(); vl_losses = []
    with torch.no_grad():
        for xb, yb, sb in vl_dl:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE); sb=sb.to(DEVICE)
            vl_losses.append(composite_loss(model_scat(xb,sb), yb).item())

    tr_l = np.mean(tr_losses); vl_l = np.mean(vl_losses)
    lr   = optimizer.param_groups[0]['lr']
    history['train'].append(tr_l); history['val'].append(vl_l); history['lr'].append(lr)

    if vl_l < best_val:
        best_val = vl_l; no_improve = 0
        torch.save(model_scat.state_dict(), CKPT_PATH)
    else:
        no_improve += 1

    pbar.set_postfix(tr=f'{tr_l:.4f}', vl=f'{vl_l:.4f}', lr=f'{lr:.1e}', pat=no_improve)
    if no_improve >= patience:
        tqdm.write(f'Early stop at ep {ep}'); break

model_scat.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
print(f"\n✅ Best val loss: {best_val:.5f}  checkpoint: {CKPT_PATH}")

In [ ]:
# CELL 15 — TRAINING HISTORY
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history['train'])+1)

axes[0].plot(ep, history['train'], label='Train', color='#e74c3c', lw=2)
axes[0].plot(ep, history['val'],   label='Val',   color='#2ecc71', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Composite Loss')
axes[0].set_title('ECGUNetScat — Training Loss', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogy(ep, history['lr'], color='#3498db', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Learning Rate')
axes[1].set_title('CAWR Learning Rate Schedule', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.suptitle('ECGUNetScat Training with Scattering Features (FiLM)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '05_training_history_scat.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ 05_training_history_scat.png saved")

In [ ]:
# CELL 16 — EVALUATE ON TEST SET & COMPARE
from torch.utils.data import TensorDataset, DataLoader as DL

mu_xt, si_xt, X_te_n = fit_norm(X_test)
X_te_n = (X_test - mu_x) / (si_x + 1e-8)
y_te_n = (y_test - mu_y) / (si_y + 1e-8)
X_te_t  = torch.tensor(X_te_n.transpose(0,2,1), dtype=torch.float32)
y_te_t  = torch.tensor(y_te_n,                   dtype=torch.float32)
sf_te_t = torch.tensor(scat_test_n,             dtype=torch.float32)
te_dl = DL(TensorDataset(X_te_t, y_te_t, sf_te_t), batch_size=64, shuffle=False)

model_scat.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb, sb in te_dl:
        xb=xb.to(DEVICE); sb=sb.to(DEVICE)
        all_preds.append(model_scat(xb, sb).cpu().numpy())
        all_true.append(yb.numpy())

preds_np = np.concatenate(all_preds) * si_y + mu_y
truth_np = np.concatenate(all_true)  * si_y + mu_y

rmse = float(np.sqrt(((preds_np - truth_np)**2).mean()))
mae  = float(np.abs(preds_np - truth_np).mean())

print(f"ECGUNetScat (with scattering features):")
print(f"  Test RMSE : {rmse:.5f} mV")
print(f"  Test MAE  : {mae:.5f}  mV")

fig, axes = plt.subplots(3, 4, figsize=(18, 9))
t_fc = np.arange(HORIZON) / FS
for r in range(3):
    for c, li in enumerate([0,1,6,11]):
        ax = axes[r, c]
        ax.plot(t_fc, truth_np[r, :, li],  color='#2980b9', lw=2,   label='Actual')
        ax.plot(t_fc, preds_np[r, :, li],  color='#e74c3c', lw=1.8, ls='--', label='ECGUNetScat')
        ax.set_title(f'S{r} Lead {LEAD_NAMES[li]}', fontsize=9, fontweight='bold')
        ax.grid(alpha=0.3)
        if r==0 and c==0: ax.legend(fontsize=8)

plt.suptitle(f'ECGUNetScat Test Predictions  (RMSE={rmse:.4f} mV)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '06_scat_test_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ 06_scat_test_predictions.png saved")

In [ ]:
# CELL 17 — FEATURE IMPORTANCE OF SCATTERING COEFFICIENTS
model_scat.eval()
N_PROBE = min(200, len(X_te_t))
xb  = X_te_t[:N_PROBE].to(DEVICE)
yb  = y_te_t[:N_PROBE].to(DEVICE)
sb  = sf_te_t[:N_PROBE].to(DEVICE).requires_grad_(True)

pred = model_scat(xb, sb)
loss = F.mse_loss(pred, yb)
loss.backward()

scat_grad = sb.grad.detach().cpu().numpy()
scat_imp  = np.abs(scat_grad).mean(0)

per_lead_imp = scat_imp.reshape(N_LEADS, -1).mean(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(scat_imp, color='#8e44ad', lw=1.5)
axes[0].fill_between(range(len(scat_imp)), 0, scat_imp, alpha=0.25, color='#8e44ad')
axes[0].set_xlabel('Scattering Coefficient Index', fontsize=11)
axes[0].set_ylabel('Mean |Gradient|', fontsize=11)
axes[0].set_title('Scattering Feature Importance — All Coefficients', fontweight='bold', fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].bar(LEAD_NAMES, per_lead_imp, color='#8e44ad', alpha=0.85, edgecolor='black')
axes[1].set_ylabel('Mean |Gradient| (averaged over scattering coefs)', fontsize=10)
axes[1].set_title('Per-Lead Scattering Importance', fontweight='bold', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Scattering Feature Gradient Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '07_scat_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ 07_scat_feature_importance.png saved")

In [ ]:
# CELL 18 — SUMMARY
print("=" * 60)
print("  07 — Wavelet & Scattering Transform  SUMMARY")
print("=" * 60)
print()
print("Wavelet families studied:")
print("  CWT (Morlet)  DWT (db4)  SWT  Haar  Sym5  Coif3")
print()
print(f"Scattering transform config: J={J}  Q={Q}  input={INPUT_LEN}")
print(f"Scattering feature dim per sample: {SCAT_DIM}")
print(f"  = {N_COEF} coefficients × {N_LEADS} leads (time-averaged)")
print()
print("Feature injection: FiLM (Feature-wise Linear Modulation)")
print("  static scat vector → gamma + beta → conditions bottleneck")
print()
print(f"ECGUNetScat Test RMSE : {rmse:.5f} mV")
print(f"ECGUNetScat Test MAE  : {mae:.5f}  mV")
print()
print("Files saved:")
print(f"  Figures  → {FIG_DIR}")
print(f"  Features → {FEAT_DIR}/scat_[train|val|test]_norm.npy")
print(f"  Model    → {CKPT_PATH}")
print()
print("✅ Notebook 07 complete")